# PJM Solar Forecast — DA Cutoff Revision Analysis

Inspects how PJM solar forecasts for the next 48 hours evolve leading up to the DA market window.
The last forecast before 9am EST is the cutoff vintage a trader uses for DA bidding.
We compare the cutoff forecast to 6h / 12h / 24h earlier vintages to reveal
revision direction, magnitude, and patterns across forward dates.

Tracks both **grid-scale solar** and **behind-the-meter (BTM) solar**.

## 1. Setup & Data Pull

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

REPO_ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(REPO_ROOT / "backend"))
from src.utils.azure_postgresql import pull_from_db

INFO:root:CONFIG_DIR: c:\Users\AidanKeaveny\Documents\github\helioscta-pjm-da\backend\src


In [2]:
# Load the cutoff analysis query
sql_path = Path.cwd() / "pjm_solar_forecast_da_cutoff.sql"
query = sql_path.read_text()
df = pull_from_db(query=query)

print(f"{len(df):,} rows")
print(f"Date range: {df['forecast_date'].min()} to {df['forecast_date'].max()}")
df.head(10)

48 rows
Date range: 2026-03-04 to 2026-03-05


,forecast_date,hour_ending,cutoff_solar_mw,cutoff_solar_btm_mw,cutoff_execution_dt,exec_dt_6h,solar_mw_6h,solar_btm_mw_6h,exec_dt_12h,solar_mw_12h,solar_btm_mw_12h,exec_dt_24h,solar_mw_24h,solar_btm_mw_24h,delta_6h,delta_12h,delta_24h,delta_btm_6h,delta_btm_12h,delta_btm_24h
0,2026-03-05,1.0,0.000,0.000,2026-03-04 08:00:00,2026-03-04 02:00:00,0.000,0.000,NaT,NaN,NaN,NaT,NaN,NaN,0.000,NaN,NaN,0.000,NaN,NaN
1,2026-03-05,2.0,0.000,0.000,2026-03-04 08:00:00,2026-03-04 02:00:00,0.000,0.000,NaT,NaN,NaN,NaT,NaN,NaN,0.000,NaN,NaN,0.000,NaN,NaN
2,2026-03-05,3.0,0.000,0.000,2026-03-04 08:00:00,2026-03-04 02:00:00,0.000,0.000,NaT,NaN,NaN,NaT,NaN,NaN,0.000,NaN,NaN,0.000,NaN,NaN
3,2026-03-05,4.0,0.000,0.000,2026-03-04 08:00:00,2026-03-04 02:00:00,0.000,0.000,NaT,NaN,NaN,NaT,NaN,NaN,0.000,NaN,NaN,0.000,NaN,NaN
4,2026-03-05,5.0,0.000,0.000,2026-03-04 08:00:00,2026-03-04 02:00:00,0.000,0.000,NaT,NaN,NaN,NaT,NaN,NaN,0.000,NaN,NaN,0.000,NaN,NaN
5,2026-03-05,6.0,0.000,0.000,2026-03-04 08:00:00,2026-03-04 02:00:00,0.000,0.000,NaT,NaN,NaN,NaT,NaN,NaN,0.000,NaN,NaN,0.000,NaN,NaN
6,2026-03-05,7.0,17.713,55.458,2026-03-04 08:00:00,2026-03-04 02:00:00,15.832,16.829,NaT,NaN,NaN,NaT,NaN,NaN,1.881,NaN,NaN,38.629,NaN,NaN
7,2026-03-05,8.0,858.055,470.004,2026-03-04 08:00:00,2026-03-04 02:00:00,805.725,587.600,NaT,NaN,NaN,NaT,NaN,NaN,52.330,NaN,NaN,-117.596,NaN,NaN
8,2026-03-05,9.0,3054.398,920.656,2026-03-04 08:00:00,2026-03-04 02:00:00,2898.733,1149.410,NaT,NaN,NaN,NaT,NaN,NaN,155.665,NaN,NaN,-228.754,NaN,NaN
9,2026-03-05,10.0,4523.257,1275.582,2026-03-04 08:00:00,2026-03-04 02:00:00,4355.275,1587.407,NaT,NaN,NaN,NaT,NaN,NaN,167.982,NaN,NaN,-311.825,NaN,NaN


## 1b. Forecast Coverage Summary

One row per forecast date showing the cutoff vintage used, each lookback vintage,
and peak cutoff solar — a quick reference for which days have full 24h→6h history.

In [3]:
_fmt_dt = lambda s: pd.to_datetime(s).strftime("%Y-%m-%d %H:%M") if pd.notna(s) else "—"

forecast_summary = (
    df.groupby("forecast_date")
    .agg(
        cutoff_exec=("cutoff_execution_dt", "first"),
        exec_6h=("exec_dt_6h", "first"),
        exec_12h=("exec_dt_12h", "first"),
        exec_24h=("exec_dt_24h", "first"),
        peak_cutoff_mw=("cutoff_solar_mw", "max"),
        min_cutoff_mw=("cutoff_solar_mw", "min"),
        hours_covered=("hour_ending", "nunique"),
    )
    .reset_index()
    .sort_values("forecast_date")
)

forecast_summary["days_ahead"] = (
    pd.to_datetime(forecast_summary["forecast_date"]) - pd.Timestamp.now().normalize()
).dt.days

for col in ["cutoff_exec", "exec_6h", "exec_12h", "exec_24h"]:
    forecast_summary[col] = forecast_summary[col].apply(_fmt_dt)

forecast_summary["has_6h"] = forecast_summary["exec_6h"] != "—"
forecast_summary["has_12h"] = forecast_summary["exec_12h"] != "—"
forecast_summary["has_24h"] = forecast_summary["exec_24h"] != "—"

display_cols = [
    "forecast_date", "days_ahead", "hours_covered",
    "cutoff_exec", "exec_24h", "exec_12h", "exec_6h",
    "has_24h", "has_12h", "has_6h",
    "peak_cutoff_mw", "min_cutoff_mw",
]
forecast_summary[display_cols].style.set_caption(
    "Solar Forecast Date Coverage — 48h Forward Window"
).format({
    "peak_cutoff_mw": "{:,.0f}",
    "min_cutoff_mw": "{:,.0f}",
}).set_properties(**{"text-align": "center"})

,forecast_date,days_ahead,hours_covered,cutoff_exec,exec_24h,exec_12h,exec_6h,has_24h,has_12h,has_6h,peak_cutoff_mw,min_cutoff_mw
0,2026-03-04,0,24,2026-03-04 00:00,2026-03-03 00:00,2026-03-03 12:00,2026-03-03 18:00,True,True,True,"5,401",0
1,2026-03-05,1,24,2026-03-04 08:00,—,—,2026-03-04 02:00,False,False,True,"5,173",0


## 2. Data Validation — Cutoff Vintage Inspection

In [4]:
cutoff_times = (
    df.groupby("forecast_date")["cutoff_execution_dt"]
    .first()
    .reset_index()
)
cutoff_times["cutoff_hour"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.hour
cutoff_times["cutoff_minute"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.minute
cutoff_times["cutoff_time_str"] = pd.to_datetime(cutoff_times["cutoff_execution_dt"]).dt.strftime("%H:%M")

print("Cutoff execution timestamps (should all be before 09:00 EST):")
cutoff_times[["forecast_date", "cutoff_execution_dt", "cutoff_time_str"]]

Cutoff execution timestamps (should all be before 09:00 EST):


,forecast_date,cutoff_execution_dt,cutoff_time_str
0,2026-03-04,2026-03-04 00:00:00,00:00
1,2026-03-05,2026-03-04 08:00:00,08:00


In [5]:
fig = px.bar(
    cutoff_times,
    x="forecast_date",
    y="cutoff_hour",
    text="cutoff_time_str",
    title="Solar — Cutoff Vintage Hour-of-Day (EST) by Forecast Date",
    template="plotly_dark",
)
fig.add_hline(y=9, line_dash="dash", line_color="red", annotation_text="9am EST cutoff")
fig.update_layout(yaxis_title="Hour (EST)", xaxis_title="Forecast Date")
fig.show()

In [6]:
coverage = (
    df.groupby("forecast_date")
    .agg(
        has_6h=("solar_mw_6h", lambda x: x.notna().any()),
        has_12h=("solar_mw_12h", lambda x: x.notna().any()),
        has_24h=("solar_mw_24h", lambda x: x.notna().any()),
        exec_dt_6h=("exec_dt_6h", "first"),
        exec_dt_12h=("exec_dt_12h", "first"),
        exec_dt_24h=("exec_dt_24h", "first"),
    )
    .reset_index()
)
print("Lookback coverage:")
coverage

Lookback coverage:


,forecast_date,has_6h,has_12h,has_24h,exec_dt_6h,exec_dt_12h,exec_dt_24h
0,2026-03-04,True,True,True,2026-03-03 18:00:00,2026-03-03 12:00:00,2026-03-03
1,2026-03-05,True,False,False,2026-03-04 02:00:00,NaT,NaT


## 3. Forecast Evolution — Cutoff vs Lookbacks (Grid-Scale Solar)

In [7]:
latest_date = df["forecast_date"].max()
df_latest = df[df["forecast_date"] == latest_date].copy().sort_values("hour_ending")

colors = {"24h": "#636EFA", "12h": "#EF553B", "6h": "#00CC96", "Cutoff": "#FFA15A"}
dashes = {"24h": "dot", "12h": "dash", "6h": "dashdot", "Cutoff": "solid"}

fig = go.Figure()
for label, col, width in [
    ("24h", "solar_mw_24h", 1),
    ("12h", "solar_mw_12h", 1.5),
    ("6h", "solar_mw_6h", 2),
    ("Cutoff", "cutoff_solar_mw", 3),
]:
    fig.add_trace(
        go.Scatter(
            x=df_latest["hour_ending"],
            y=df_latest[col],
            name=label,
            line=dict(color=colors[label], dash=dashes[label], width=width),
        )
    )

fig.update_layout(
    height=450,
    template="plotly_dark",
    title=f"Solar — Forecast Evolution — Latest Date ({latest_date})",
    xaxis_title="Hour Ending",
    yaxis_title="Solar (MW)",
)
fig.show()

In [8]:
# All dates — cutoff (solid) vs 12h lookback (dashed)
dates = sorted(df["forecast_date"].unique())

fig = go.Figure()
for d in dates:
    ddf = df[df["forecast_date"] == d].sort_values("hour_ending")
    opacity = 0.3 if d != latest_date else 1.0
    fig.add_trace(go.Scatter(
        x=ddf["hour_ending"], y=ddf["cutoff_solar_mw"],
        name=f"{d} cutoff",
        line=dict(width=2 if d == latest_date else 1),
        opacity=opacity,
        showlegend=True,
    ))
    fig.add_trace(go.Scatter(
        x=ddf["hour_ending"], y=ddf["solar_mw_12h"],
        name=f"{d} 12h",
        line=dict(dash="dash", width=1),
        opacity=opacity * 0.6,
        showlegend=False,
    ))

fig.update_layout(
    height=500, template="plotly_dark",
    title="Solar — Cutoff (solid) vs 12h Prior (dashed)",
    xaxis_title="Hour Ending", yaxis_title="Solar (MW)",
)
fig.show()

## 4. Forecast Deltas — MW Changes at Each Lookback

In [9]:
fig = go.Figure()
for col, label, color in [
    ("delta_24h", "24h→Cutoff", "#636EFA"),
    ("delta_12h", "12h→Cutoff", "#EF553B"),
    ("delta_6h", "6h→Cutoff", "#00CC96"),
]:
    fig.add_trace(go.Bar(
        x=df_latest["hour_ending"], y=df_latest[col],
        name=label, marker_color=color,
    ))

fig.update_layout(
    barmode="group",
    title=f"Solar — Forecast Revision Deltas ({latest_date})",
    xaxis_title="Hour Ending", yaxis_title="Delta (MW)",
    template="plotly_dark", height=400,
)
fig.add_hline(y=0, line_color="white", line_width=0.5)
fig.show()

In [10]:
pivot_12h = df.pivot_table(
    index="forecast_date", columns="hour_ending", values="delta_12h",
)

fig = px.imshow(
    pivot_12h.values,
    x=[f"HE{int(c)}" for c in pivot_12h.columns],
    y=[str(d) for d in pivot_12h.index],
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    aspect="auto",
    title="Solar — 12h Forecast Revision (MW) by Date × Hour",
    labels={"color": "Delta MW"},
    template="plotly_dark",
)
fig.update_layout(height=500)
fig.show()

## 5. Per-Date Revision Deep Dive

For each forecast date, show the 24h → 12h → 6h → cutoff evolution
so we can inspect how each day's solar forecast converged.

In [11]:
dates = sorted(df["forecast_date"].unique())

fig = make_subplots(
    rows=len(dates), cols=1,
    shared_xaxes=True,
    subplot_titles=[f"Solar — {d}" for d in dates],
    vertical_spacing=0.04,
)

for i, d in enumerate(dates, 1):
    ddf = df[df["forecast_date"] == d].sort_values("hour_ending")

    for label, col, exec_col, width in [
        ("24h", "solar_mw_24h", "exec_dt_24h", 1),
        ("12h", "solar_mw_12h", "exec_dt_12h", 1.5),
        ("6h", "solar_mw_6h", "exec_dt_6h", 2),
        ("Cutoff", "cutoff_solar_mw", "cutoff_execution_dt", 3),
    ]:
        exec_dts = ddf[exec_col].astype(str).values
        forecast_dates = ddf["forecast_date"].astype(str).values
        if label == "Cutoff":
            deltas = np.zeros(len(ddf))
        else:
            delta_col_name = f"delta_{label}"
            deltas = ddf[delta_col_name].values
        rank_labels = np.array([label] * len(ddf))

        customdata = np.column_stack([exec_dts, forecast_dates, deltas, rank_labels])

        fig.add_trace(
            go.Scatter(
                x=ddf["hour_ending"],
                y=ddf[col],
                name=label,
                line=dict(color=colors[label], dash=dashes[label], width=width),
                showlegend=(i == 1),
                customdata=customdata,
                hovertemplate=(
                    "<b>%{customdata[3]}</b><br>"
                    "Forecast Date: %{customdata[1]}<br>"
                    "HE: %{x}<br>"
                    "Solar: %{y:,.0f} MW<br>"
                    "Execution DT: %{customdata[0]}<br>"
                    "Delta to Cutoff: %{customdata[2]} MW"
                    "<extra></extra>"
                ),
            ),
            row=i, col=1,
        )

fig.update_layout(
    height=250 * len(dates),
    template="plotly_dark",
    title_text="Solar — Forecast Evolution by Date",
)
fig.update_yaxes(title_text="Solar (MW)")
fig.update_xaxes(title_text="Hour Ending", row=len(dates), col=1)
fig.show()

## 6. Revision Summary Statistics

In [12]:
summary_rows = []
for lookback, col in [("6h", "delta_6h"), ("12h", "delta_12h"), ("24h", "delta_24h")]:
    vals = df[col].dropna()
    summary_rows.append({
        "Lookback": lookback,
        "Mean (MW)": vals.mean(),
        "Median (MW)": vals.median(),
        "Std (MW)": vals.std(),
        "Min (MW)": vals.min(),
        "Max (MW)": vals.max(),
        "N": len(vals),
    })

summary_df = pd.DataFrame(summary_rows)
summary_df

,Lookback,Mean (MW),Median (MW),Std (MW),Min (MW),Max (MW),N
0,6h,6.760812,0.0000,84.393259,-275.374,210.084,48
1,12h,-73.168750,-1.2915,102.871534,-334.296,0.000,24
2,24h,-87.600500,0.0000,142.626762,-449.188,5.477,24


In [13]:
date_summary = (
    df.groupby("forecast_date")
    .agg(
        mean_delta_6h=("delta_6h", "mean"),
        mean_delta_12h=("delta_12h", "mean"),
        mean_delta_24h=("delta_24h", "mean"),
        peak_cutoff_mw=("cutoff_solar_mw", "max"),
    )
    .reset_index()
)

fig = go.Figure()
for col, label, color in [
    ("mean_delta_24h", "24h", "#636EFA"),
    ("mean_delta_12h", "12h", "#EF553B"),
    ("mean_delta_6h", "6h", "#00CC96"),
]:
    fig.add_trace(go.Bar(
        x=date_summary["forecast_date"], y=date_summary[col],
        name=label, marker_color=color,
    ))

fig.update_layout(
    barmode="group",
    title="Solar — Mean Forecast Revision (MW) by Date & Lookback",
    xaxis_title="Forecast Date", yaxis_title="Avg Delta (MW)",
    template="plotly_dark", height=450,
)
fig.add_hline(y=0, line_color="white", line_width=0.5)
fig.show()

In [14]:
for lookback, delta_col in [("6h", "delta_6h"), ("12h", "delta_12h"), ("24h", "delta_24h")]:
    pivot = df.pivot_table(
        index="forecast_date", columns="hour_ending", values=delta_col,
    )
    if pivot.empty:
        continue

    fig = px.imshow(
        pivot.values,
        x=[f"HE{int(c)}" for c in pivot.columns],
        y=[str(d) for d in pivot.index],
        color_continuous_scale="RdBu",
        color_continuous_midpoint=0,
        aspect="auto",
        title=f"Solar — {lookback} Forecast Revision (MW) by Date × Hour",
        labels={"color": "Delta MW"},
        template="plotly_dark",
    )
    fig.update_layout(height=400)
    fig.show()

## 7. Behind-the-Meter (BTM) Solar — Cutoff vs Lookbacks

In [15]:
# BTM evolution for latest date
fig = go.Figure()
for label, col, width in [
    ("24h", "solar_btm_mw_24h", 1),
    ("12h", "solar_btm_mw_12h", 1.5),
    ("6h", "solar_btm_mw_6h", 2),
    ("Cutoff", "cutoff_solar_btm_mw", 3),
]:
    fig.add_trace(
        go.Scatter(
            x=df_latest["hour_ending"],
            y=df_latest[col],
            name=label,
            line=dict(color=colors[label], dash=dashes[label], width=width),
        )
    )

fig.update_layout(
    height=450,
    template="plotly_dark",
    title=f"BTM Solar — Forecast Evolution — Latest Date ({latest_date})",
    xaxis_title="Hour Ending",
    yaxis_title="BTM Solar (MW)",
)
fig.show()

In [16]:
# BTM deltas for latest date
fig = go.Figure()
for col, label, color in [
    ("delta_btm_24h", "24h→Cutoff", "#636EFA"),
    ("delta_btm_12h", "12h→Cutoff", "#EF553B"),
    ("delta_btm_6h", "6h→Cutoff", "#00CC96"),
]:
    fig.add_trace(go.Bar(
        x=df_latest["hour_ending"], y=df_latest[col],
        name=label, marker_color=color,
    ))

fig.update_layout(
    barmode="group",
    title=f"BTM Solar — Forecast Revision Deltas ({latest_date})",
    xaxis_title="Hour Ending", yaxis_title="Delta (MW)",
    template="plotly_dark", height=400,
)
fig.add_hline(y=0, line_color="white", line_width=0.5)
fig.show()